# Quickstart Reproduction: TEM Virus Multiprotocol

This notebook reproduces the lightweight, non-training parts of the study using artifacts that already live in this repository: the source-aware split manifests, the recorded per-seed and aggregate result tables, and the released DenseNet201 weights. It is designed to run from start to finish with **Run All** on a free Google Colab runtime, and it completes in a few minutes.

It does **not** retrain any model. The full study trains two architectures (DenseNet201 and EfficientNetV2-S) across four protocols and three to five seeds, which is many GPU-hours of compute and cannot finish inside a single free runtime. That path is documented in `REPRODUCE.md` and the command line tools in `scripts/`, and it requires a GPU and the image dataset.

The original experiment notebooks in this folder are kept as the research log. They record the full exploration, including configuration trials, resume and recovery steps, and stale outputs, and they are tied to the original Colab environment. This quickstart is the clean, verifiable entry point that sits alongside them.

## What this notebook covers

The sections below run in order and depend only on the cloned repository and a single weights download. No local paths and no Google Drive are used.

Runnable here, with no GPU and no image dataset:

1. Validation of the four source-aware split manifests.
2. Inspection of the recorded accuracy and macro F1 results.
3. Recomputation of the paired statistics, checked against the recorded values.
4. A demonstration of the ensemble combination and selection logic.
5. Loading of the released weights and a check of the inference path.

Out of scope here, because it needs a GPU and the dataset:

- Training the models and regenerating the per-seed metrics from scratch.

For full training, see `REPRODUCE.md`. For interactive classification of real images, use the live demo linked at the end.

## Section 0. Setup

The cell below clones the repository with a shallow clone, puts the `src` directory on the import path, and records the locations of the split manifests and result tables. Re-running it is safe, because it only clones when the directory is not already present.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/usmanuhaka/tem-virus-multiprotocol.git"
REPO_DIR = "tem-virus-multiprotocol"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

SPLITS_DIR = os.path.join(REPO_DIR, "data", "splits")
SUMMARIES_DIR = os.path.join(REPO_DIR, "results", "summaries")

import temvirus
print("temvirus version:", temvirus.__version__)
print("repository path:", os.path.abspath(REPO_DIR))

## Section 1. Validate the source-aware splits

The four evaluation protocols are distributed as split manifests, one row per image crop, under `data/splits/`. These manifests are the canonical partitions: anyone who reproduces the study uses exactly the same train, validation, and test assignments without rerunning the dataset-dependent split construction.

This step re-checks each manifest for the properties it must satisfy:

- the train, validation, and test sizes match the values reported in the paper,
- all 14 classes appear in every split,
- each `label_id` is consistent with the alphabetical class order,
- no source group spans more than one split, where the group is the RAW source image for Protocol B-G14 and the acquisition session for Protocol C-G09,
- no single crop, identified by its full file path, appears in more than one split.

Protocols A and A-clean-strict use the dataset's native crop-level split, so they are not expected to be source-grouped, and the group-spanning check applies to B-G14 and C-G09. The check reads only the manifests, so it needs no images and no GPU.

In [ ]:
import pandas as pd
from temvirus.config import PROTOCOL_MANIFEST
from temvirus.splits import validate_manifest

rows = []
for protocol, fname in PROTOCOL_MANIFEST.items():
    rep = validate_manifest(os.path.join(SPLITS_DIR, fname), protocol)
    rows.append({
        "protocol": protocol,
        "train_val_test": rep["sizes"],
        "n_classes": rep["n_classes"],
        "groups_spanning_splits": rep.get("groups_spanning_splits", "n/a"),
        "crop_overlaps": rep["filepath_split_overlaps"],
        "ok": rep["ok"],
    })

report = pd.DataFrame(rows)
print("All manifests valid:", bool(report["ok"].all()))
report

## Section 2. Recorded results

The repository ships the recorded evaluation outputs so the headline numbers can be inspected without rerunning training. Two files are loaded here:

- `CP4_ensemble_aggregate_by_protocol.csv` holds the per-protocol means and standard deviations across seeds. Its accuracy columns correspond to Table 1 in the paper and its macro F1 columns to Table 2.
- `CP4_ensemble_seed_level.csv` holds the per-seed test accuracy for the baseline DenseNet201, the single models, and the ensemble, together with the ensemble strategy that was selected on the validation split for each seed. That strategy column corresponds to the distribution reported in Table 4.

The views below are curated subsets for readability. The full files, along with the per-class results and the confusion-matrix summaries, are available under `results/`.

In [ ]:
agg = pd.read_csv(os.path.join(SUMMARIES_DIR, "CP4_ensemble_aggregate_by_protocol.csv"))
seed_level = pd.read_csv(os.path.join(SUMMARIES_DIR, "CP4_ensemble_seed_level.csv"))

accuracy_view = agg[[
    "display_name",
    "baseline_accuracy_mean", "densenet_accuracy_mean",
    "effnet_accuracy_mean", "ensemble_accuracy_mean", "ensemble_best_seed_accuracy",
]].round(4)
print("Mean test accuracy by method and protocol:")
accuracy_view

In [ ]:
seed_view = seed_level[[
    "protocol", "seed",
    "densenet_test_accuracy", "ensemble_test_accuracy",
    "best_strategy_by_val_macro_f1",
]].round(4)
print("Per-seed test accuracy and the validation-selected ensemble strategy:")
seed_view

## Section 3. Recompute the paired statistics

The paper compares methods within matched protocol and seed settings using paired statistics: the mean accuracy difference, a Wilcoxon signed-rank p-value, Cohen's d, and a 95 percent confidence interval from 10,000 bootstrap resamples, with significance judged against a Bonferroni threshold of 0.05 divided by 12.

Because the seed-level table carries the per-seed DenseNet201 and ensemble accuracies, the Ensemble versus TTA-plus-mixup DenseNet201 comparison can be recomputed here directly and checked against the recorded `statistical_tests_results.csv`. The recompute uses the same functions that produced the recorded file, so the values are expected to match.

One caveat is built into the design. With three to five seeds, the smallest attainable two-sided signed-rank p-value is 0.25 or 0.0625, so it cannot cross the Bonferroni threshold. The interpretation therefore rests on the effect size and the bootstrap interval rather than on the p-value.

In [ ]:
from temvirus.stats import build_comparison_row, BONFERRONI_THRESHOLD

dn, ens = {}, {}
for _, r in seed_level.iterrows():
    dn.setdefault(str(r["protocol"]), {})[int(r["seed"])] = float(r["densenet_test_accuracy"])
    ens.setdefault(str(r["protocol"]), {})[int(r["seed"])] = float(r["ensemble_test_accuracy"])

protocols = ["A", "B_G14", "A_clean_strict", "C_G09"]
recomputed = pd.DataFrame([
    build_comparison_row(p, "Ensemble_vs_TTAMixup_DN", ens[p], dn[p]) for p in protocols
])

print("Bonferroni threshold = 0.05 / 12 =", round(BONFERRONI_THRESHOLD, 6))
recomputed[["protocol", "n_seeds", "mean_diff", "cohens_d", "wilcoxon_p", "ci_low", "ci_high"]].round(4)

In [ ]:
recorded = pd.read_csv(os.path.join(SUMMARIES_DIR, "statistical_tests_results.csv"))
recorded_sub = recorded[recorded["comparison"] == "Ensemble_vs_TTAMixup_DN"]

check = recomputed.merge(recorded_sub, on=["protocol", "comparison"], suffixes=("_recomputed", "_recorded"))
comparison = check[[
    "protocol",
    "mean_diff_recomputed", "mean_diff_recorded",
    "cohens_d_recomputed", "cohens_d_recorded",
    "ci_low_recomputed", "ci_low_recorded",
    "ci_high_recomputed", "ci_high_recorded",
]].round(5)

max_abs_diff = (comparison["mean_diff_recomputed"] - comparison["mean_diff_recorded"]).abs().max()
print("Max absolute difference in mean_diff (recomputed vs recorded):", float(max_abs_diff))
comparison

## Section 4. Ensemble combination logic

The two-model ensemble combines the softmax outputs of DenseNet201 and EfficientNetV2-S at inference time. For each protocol and seed, six strategies are scored on the validation split: each single model on its own, a simple average, a validation-F1-weighted average, a geometric mean, and a maximum-confidence rule. The strategy with the highest validation macro F1 is then applied once to the test split, so the test set never influences the choice.

The per-image softmax arrays are large and are not bundled in the repository, so the cell below demonstrates the combination and selection functions on a small illustrative example. It shows the mechanism and the selection rule rather than reproducing the reported ensemble accuracy. The selections that were actually made for each protocol and seed are the `best_strategy_by_val_macro_f1` values shown in Section 2.

In [ ]:
import numpy as np
from temvirus import ensemble as E

rng = np.random.default_rng(0)

def softmax_block(n, k=14):
    z = rng.normal(size=(n, k))
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# Illustrative validation and test softmax for the two models (synthetic, not real outputs).
effnet_val, dense_val = softmax_block(40), softmax_block(40)
effnet_test, dense_test = softmax_block(20), softmax_block(20)
y_val = rng.integers(0, 14, size=40)

best, ens_val, ens_test, val_table = E.select_and_apply(
    effnet_val, dense_val, effnet_test, dense_test, y_val
)

print("Validation macro F1 by strategy:")
for r in sorted(val_table, key=lambda row: -row["macro_f1"]):
    print("  {:>18}: {:.4f}".format(r["strategy"], r["macro_f1"]))

print("\nSelected strategy (highest validation macro F1):", best)
print("Ensembled test softmax shape:", ens_test.shape,
      "| rows sum to 1:", bool(np.allclose(ens_test.sum(axis=1), 1.0)))

## Section 5. Released weights and the inference path

The trained DenseNet201 weights for Protocol A are published as a release asset. This section downloads that checkpoint, rebuilds the exact architecture with `timm`, loads the weights, and runs the four-view test-time augmentation that the study uses at inference: the original image, a horizontal flip, a vertical flip, and both flips, with the four softmax vectors averaged.

The repository ships split manifests rather than images, since the dataset is hosted on Mendeley Data, so this section runs the inference path on a single synthetic input. The purpose is to confirm that the released checkpoint loads into the architecture and produces a valid 14-class probability vector, not to classify a real specimen. For real TEM crops, use the live demo linked at the end, or download the dataset and point `scripts/evaluate.py` at it.

This section installs `timm` and downloads about 74 MB, so it is heavier than the sections above, but it still completes on a free runtime and needs no GPU.

In [ ]:
import importlib.util, sys, subprocess

# Install timm only if it is not already available (Colab ships torch but not timm).
if importlib.util.find_spec("timm") is None:
    base = [sys.executable, "-m", "pip", "install", "-q", "timm"]
    if subprocess.run(base).returncode != 0:
        subprocess.run(base + ["--break-system-packages"], check=True)
print("timm is ready")

In [ ]:
import os, urllib.request, torch
from temvirus.config import CLASS_NAMES
from temvirus.models import build_model
from temvirus.tta import tta_predict

WEIGHTS_URL = "https://github.com/usmanuhaka/tem-virus-multiprotocol/releases/download/v1.0.0/best.pt"
weights_path = os.path.join(REPO_DIR, "weights", "best.pt")
os.makedirs(os.path.dirname(weights_path), exist_ok=True)

if not os.path.exists(weights_path):
    print("Downloading released weights (about 74 MB) ...")
    urllib.request.urlretrieve(WEIGHTS_URL, weights_path)
print("Checkpoint size:", round(os.path.getsize(weights_path) / 1e6, 1), "MB")

checkpoint = torch.load(weights_path, map_location="cpu")
state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
meta = [k for k in checkpoint if k != "model_state_dict"] if isinstance(checkpoint, dict) else []
print("Metadata keys in checkpoint:", meta)
if isinstance(checkpoint, dict):
    if checkpoint.get("epoch") is not None:
        print("Recorded best epoch:", checkpoint["epoch"])
    if checkpoint.get("val_macro_f1") is not None:
        print("Validation macro F1:", round(float(checkpoint["val_macro_f1"]), 4))

model = build_model("densenet201", pretrained=False, num_classes=len(CLASS_NAMES))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model.eval()
print("Weights loaded. Missing keys:", len(missing), "| unexpected keys:", len(unexpected))

# Inference path on one synthetic 224x224 RGB input (placeholder, not a real TEM crop).
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    probs = tta_predict(model, dummy)

top = int(probs.argmax(dim=1))
print("\nInference path OK. Output shape", tuple(probs.shape),
      "| total probability", round(float(probs.sum()), 3))
print("On the synthetic input the top index is", top, "(" + CLASS_NAMES[top] + ").")
print("This is a pipeline check. For real classification use the demo or the dataset.")

## Summary and next steps

This notebook ran the verifiable, non-training parts of the study from start to finish:

- the four split manifests were validated for size, class coverage, label consistency, and the absence of source-group and crop-level leakage,
- the recorded accuracy and macro F1 results were loaded and inspected,
- the paired statistics were recomputed from the per-seed accuracies and matched the recorded values,
- the ensemble combination and selection logic was demonstrated,
- the released DenseNet201 checkpoint was downloaded, loaded into the architecture, and exercised through the four-view inference path.

Where to go next:

- Full training across all protocols and seeds is in `scripts/` and is described step by step in `REPRODUCE.md`. It needs a GPU and the image dataset from Mendeley Data (doi:10.17632/x4dwwfwtw3.3).
- Interactive classification of real TEM crops is available in the live demo at https://temvirusmultiprotocol.streamlit.app.
- A citable archive of the repository is on Zenodo at https://doi.org/10.5281/zenodo.20794278.
- The two original notebooks in this folder are the experimental record. They document the research process and are not the reproduction entry point.